### 04_silver_to_gold_ingestion

Connect to Postgre > Create the tables for Gold layer > Load data from Silver into the tables > Validate it

In [2]:
from pathlib import Path
import sys

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

## Parameters

In [8]:
silver_dir = f"{project_root}/data/silver"
truncate_before_load = True
validation_limit = 5

## Imports

In [9]:
import pandas as pd

from src.db.postgres_client import PostgresClient
from src.db.create_tables import PostgresSchemaCreator
from src.ingestion.silver_to_gold_loader import SilverToGoldIngestion

## Test connection

In [10]:
client = PostgresClient()
client.test_connection()

Connection to PostgreSQL established successfully.


## Create Gold tables

In [11]:
creator = PostgresSchemaCreator()
creator.run()

Connection to PostgreSQL established successfully.
Creating table: dim_tempo
Table created successfully: dim_tempo
Creating table: dim_localidade
Table created successfully: dim_localidade
Creating table: dim_condicao_via
Table created successfully: dim_condicao_via
Creating table: dim_vitima
Table created successfully: dim_vitima
Creating table: dim_tipo_veiculo
Table created successfully: dim_tipo_veiculo
Creating table: fato_acidente
Table created successfully: fato_acidente
Creating table: fato_vitima
Table created successfully: fato_vitima
Creating table: fato_veiculo_acidente
Table created successfully: fato_veiculo_acidente
All PostgreSQL tables were created successfully.


## Build Gold tables from Silver and load into PostgreSQL

In [12]:
loader = SilverToGoldIngestion(silver_dir=silver_dir)
gold_tables = loader.run(truncate_before_load=truncate_before_load)

for table_name, dataframe in gold_tables.items():
    print(f"{table_name}: {len(dataframe)} row(s) built before load")

Connection to PostgreSQL established successfully.
Reading silver part: c:\Users\Diane\git\Lab01_PART1_97103\data\silver\Acidentes_DadosAbertos\part_0001.parquet
Reading silver part: c:\Users\Diane\git\Lab01_PART1_97103\data\silver\Acidentes_DadosAbertos\part_0002.parquet
Reading silver part: c:\Users\Diane\git\Lab01_PART1_97103\data\silver\Acidentes_DadosAbertos\part_0003.parquet
Reading silver part: c:\Users\Diane\git\Lab01_PART1_97103\data\silver\Acidentes_DadosAbertos\part_0004.parquet
Reading silver part: c:\Users\Diane\git\Lab01_PART1_97103\data\silver\Acidentes_DadosAbertos\part_0005.parquet
Reading silver part: c:\Users\Diane\git\Lab01_PART1_97103\data\silver\Acidentes_DadosAbertos\part_0006.parquet
Reading silver part: c:\Users\Diane\git\Lab01_PART1_97103\data\silver\Acidentes_DadosAbertos\part_0007.parquet
Reading silver part: c:\Users\Diane\git\Lab01_PART1_97103\data\silver\Acidentes_DadosAbertos\part_0008.parquet
Combined 8 parquet part(s) from Acidentes_DadosAbertos: 46436

MemoryError: Unable to allocate 1.62 GiB for an array with shape (22, 9856817) and data type object

## Validation queries

In [ ]:
validation_tables = [
    "dim_tempo",
    "dim_localidade",
    "dim_condicao_via",
    "dim_vitima",
    "dim_tipo_veiculo",
    "fato_acidente",
    "fato_vitima",
    "fato_veiculo_acidente",
]

for table_name in validation_tables:
    print(f"\n--- {table_name} ---")
    display(loader.validate_table(table_name=table_name, limit=validation_limit))

## Optional row count check

In [ ]:
for table_name in validation_tables:
    query = f"SELECT COUNT(*) AS total_rows FROM {client.schema}.{table_name}"
    with client.get_connection() as conn:
        count_df = pd.read_sql(query, conn)
    print(f"{table_name}: {int(count_df.loc[0, 'total_rows'])}")